# 🏦 Loan Default Prediction Machine Learning Project
### An End-to-End Predictive Modeling Pipeline for Credit Risk Assessment

---
## Project Overview
In retail banking and peer-to-peer lending, identifying borrowers likely to default on their loan obligations is paramount to credit risk management. 
This notebook implements an **end-to-end Machine Learning solution in pure Python** to predict loan default (`loan_status` $\in \{0, 1\}$).

### Notebook Pipeline Order:
1. **Imports & Global Configuration**
2. **Loading Dataset**
3. **Exploratory Data Analysis (EDA)**
4. **Data Cleaning**
5. **Data Leakage Prevention & Column Dropping**
6. **Feature Engineering (Vintage & Credit Maturity)**
7. **Preprocessing & Missing Value Imputation**
8. **Stratified Train / Validation Split**
9. **Model Training (Logistic Regression, Random Forest, CatBoost)**
10. **Model Comparison & Performance Benchmarks**
11. **Comprehensive Evaluation (ROC-AUC, PR Curves, Confusion Matrices)**
12. **Feature Importance Analysis**
13. **Final Model Retraining on Full Dataset**
14. **Test Set Prediction**
15. **Submission Generation & Verification**

## 1. Imports & Environment Setup
We import core data processing libraries (`pandas`, `numpy`), visualization tools (`matplotlib`, `seaborn`), machine learning estimators (`scikit-learn`), and the gradient boosting framework (`catboost`).

In [ ]:
import os
import sys
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve,
    classification_report
)

from catboost import CatBoostClassifier, Pool
import joblib

# Visualization setup
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('All libraries successfully loaded!')

## 2. Load Dataset
We load `loan_train.csv` (27,003 records) and `loan_test.csv` (11,574 records).

In [ ]:
train_path = '../data/loan_train.csv' if os.path.exists('../data/loan_train.csv') else 'data/loan_train.csv'
test_path = '../data/loan_test.csv' if os.path.exists('../data/loan_test.csv') else 'data/loan_test.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f'Train shape: {df_train.shape}')
print(f'Test shape : {df_test.shape}')
df_train.head(3)

## 3. Exploratory Data Analysis (EDA)
We explore target balance, missing value patterns, and credit risk distributions.

In [ ]:
# Target Distribution
target_counts = df_train['loan_status'].value_counts()
print('Target Counts:')
print(target_counts)

plt.figure(figsize=(7, 4.5))
ax = sns.countplot(x='loan_status', hue='loan_status', data=df_train, palette=['#2ecc71', '#e74c3c'], legend=False)
plt.title('Target Variable Distribution (0: Fully Paid, 1: Charged Off / Default)', fontsize=13, fontweight='bold')
plt.xlabel('Loan Status')
plt.ylabel('Borrower Count')
plt.xticks([0, 1], ['Non-Default (0)', 'Default (1)'])
plt.show()

In [ ]:
# Missing Values Summary
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
missing_df.head(10)

## 4 & 5. Data Cleaning & Data Leakage Removal

### Anti-Leakage Rules:
Post-loan repayment attributes (`out_prncp`, `total_pymnt`, `recoveries`, `last_pymnt_amnt`, etc.) occur **after** a loan is issued and must be removed to avoid target leakage.

In [ ]:
LEAKAGE_COLUMNS = [
    'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee', 'last_pymnt_d',
    'last_pymnt_amnt', 'last_credit_pull_d'
]
DROP_COLUMNS = ['id', 'member_id', 'url', 'desc']
HIGH_CARD_TEXT = ['emp_title', 'title', 'zip_code']

def clean_dataset(df, is_train=True):
    df = df.copy()
    drop_cols = [c for c in LEAKAGE_COLUMNS + DROP_COLUMNS + HIGH_CARD_TEXT if c in df.columns]
    df = df.drop(columns=drop_cols)
    
    if 'int_rate' in df.columns and not pd.api.types.is_numeric_dtype(df['int_rate']):
        df['int_rate'] = df['int_rate'].astype(str).str.rstrip('%').str.strip()
        df['int_rate'] = pd.to_numeric(df['int_rate'], errors='coerce')
        
    if 'revol_util' in df.columns and not pd.api.types.is_numeric_dtype(df['revol_util']):
        df['revol_util'] = df['revol_util'].astype(str).str.rstrip('%').str.strip()
        df['revol_util'] = pd.to_numeric(df['revol_util'], errors='coerce')
        
    if 'term' in df.columns and not pd.api.types.is_numeric_dtype(df['term']):
        df['term'] = df['term'].astype(str).str.extract(r'(\d+)')[0]
        df['term'] = pd.to_numeric(df['term'], errors='coerce')
        
    if 'emp_length' in df.columns:
        def parse_emp(v):
            if pd.isna(v) or str(v).lower() == 'nan': return np.nan
            s = str(v).strip()
            if '<' in s: return 0.0
            if '10+' in s: return 10.0
            m = re.search(r'(\d+)', s)
            return float(m.group(1)) if m else np.nan
        df['emp_length'] = df['emp_length'].apply(parse_emp)
        
    def parse_dt(col):
        dt = pd.to_datetime(col, format='%b-%y', errors='coerce')
        dt = dt.map(lambda d: d.replace(year=d.year - 100) if pd.notna(d) and d.year > 2025 else d)
        return dt
        
    if 'issue_d' in df.columns:
        issue_dt = parse_dt(df['issue_d'])
        df['issue_year'] = issue_dt.dt.year
        df['issue_month'] = issue_dt.dt.month
        df = df.drop(columns=['issue_d'])
        
    if 'earliest_cr_line' in df.columns:
        cr_dt = parse_dt(df['earliest_cr_line'])
        earliest_cr_year = cr_dt.dt.year
        df['credit_history_year'] = earliest_cr_year
        if 'issue_year' in df.columns:
            df['credit_history_length_years'] = (df['issue_year'] - earliest_cr_year).clip(lower=0)
        df = df.drop(columns=['earliest_cr_line'])
        
    return df

cleaned_train = clean_dataset(df_train, is_train=True)
print(f'Cleaned training features shape: {cleaned_train.shape}')
cleaned_train.head(3)

## 6 & 7. Feature Engineering & Preprocessing Pipeline
We define a reusable `LoanPreprocessor` that handles missing values and encodings consistently across training, validation, and test datasets.

In [ ]:
class LoanPreprocessor:
    def __init__(self):
        self.numeric_cols = []
        self.categorical_cols = []
        self.numeric_medians = {}
        self.sklearn_pipeline = None
        self.feature_names = []

    def fit(self, X: pd.DataFrame):
        self.numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.categorical_cols = [c for c in X.columns if c not in self.numeric_cols]

        for col in self.numeric_cols:
            num_series = pd.to_numeric(X[col], errors='coerce')
            med = num_series.median()
            self.numeric_medians[col] = 0.0 if pd.isna(med) else float(med)

        self.feature_names = list(X.columns)

        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])

        self.sklearn_pipeline = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_cols),
                ('cat', categorical_transformer, self.categorical_cols)
            ]
        )
        self.sklearn_pipeline.fit(X)
        return self

    def transform_catboost(self, X: pd.DataFrame) -> pd.DataFrame:
        X_out = X.copy()
        for col in self.feature_names:
            if col not in X_out.columns:
                X_out[col] = self.numeric_medians.get(col, 0.0) if col in self.numeric_cols else 'Unknown'

        X_out = X_out[self.feature_names]
        for col in self.numeric_cols:
            X_out[col] = pd.to_numeric(X_out[col], errors='coerce').fillna(self.numeric_medians.get(col, 0.0))

        for col in self.categorical_cols:
            X_out[col] = X_out[col].fillna('Unknown').astype(str).replace({'nan': 'Unknown', '': 'Unknown'})

        return X_out

    def transform_sklearn(self, X: pd.DataFrame):
        X_cb = self.transform_catboost(X)
        return self.sklearn_pipeline.transform(X_cb)

## 8. Stratified Train / Validation Split
We split the training data (80% train, 20% validation) using stratification on `loan_status`.

In [ ]:
X = cleaned_train.drop(columns=['loan_status'])
y = cleaned_train['loan_status'].astype(int)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = LoanPreprocessor()
preprocessor.fit(X_train_raw)

X_train_cb = preprocessor.transform_catboost(X_train_raw)
X_val_cb = preprocessor.transform_catboost(X_val_raw)

X_train_sk = preprocessor.transform_sklearn(X_train_raw)
X_val_sk = preprocessor.transform_sklearn(X_val_raw)

print(f'Training feature matrix (CatBoost) : {X_train_cb.shape}')
print(f'Validation feature matrix (CatBoost): {X_val_cb.shape}')
print(f'One-hot feature matrix (Sklearn)   : {X_train_sk.shape}')

## 9 & 10. Model Training & Comparison
We train and benchmark:
1. **Logistic Regression** (L2 penalty, class_weight='balanced')
2. **Random Forest** (200 trees, max_depth=12, class_weight='balanced')
3. **CatBoost Classifier** (iterations=700, depth=7, learning_rate=0.05, AUC optimization)

In [ ]:
# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_sk, y_train)
lr_prob = lr_model.predict_proba(X_val_sk)[:, 1]
lr_pred = (lr_prob >= 0.5).astype(int)

# 2. Random Forest
rf_model = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train_sk, y_train)
rf_prob = rf_model.predict_proba(X_val_sk)[:, 1]
rf_pred = (rf_prob >= 0.5).astype(int)

# 3. CatBoost Classifier
cat_indices = [X_train_cb.columns.get_loc(c) for c in preprocessor.categorical_cols]
train_pool = Pool(X_train_cb, y_train, cat_features=cat_indices)
val_pool = Pool(X_val_cb, y_val, cat_features=cat_indices)

cb_model = CatBoostClassifier(
    iterations=700,
    depth=7,
    learning_rate=0.05,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50
)
cb_model.fit(train_pool, eval_set=val_pool, use_best_model=True)
cb_prob = cb_model.predict_proba(X_val_cb)[:, 1]
cb_pred = (cb_prob >= 0.5).astype(int)

## 11. Model Evaluation & Benchmark
Because loan defaults are imbalanced, Accuracy alone is deceptive. We evaluate using **ROC-AUC, Precision, Recall, and F1 Score**.

In [ ]:
def eval_metrics(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

results_df = pd.DataFrame([
    eval_metrics('Logistic Regression', y_val, lr_pred, lr_prob),
    eval_metrics('Random Forest', y_val, rf_pred, rf_prob),
    eval_metrics('CatBoost', y_val, cb_pred, cb_prob)
]).set_index('Model')

display(results_df.style.highlight_max(axis=0, color='#dcfce7'))

In [ ]:
# ROC Curves Plot
plt.figure(figsize=(8, 6))
for name, prob in [('Logistic Regression', lr_prob), ('Random Forest', rf_prob), ('CatBoost', cb_prob)]:
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc_val = roc_auc_score(y_val, prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc_val:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guess (AUC = 0.50)')
plt.title('Validation ROC Curves Comparison', fontsize=13, fontweight='bold')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.legend(loc='lower right')
plt.show()

## 12. Feature Importance Analysis
Feature importance derived from the gradient boosted trees identifies the primary determinants of borrower default.

In [ ]:
importances = cb_model.get_feature_importance()
feat_df = pd.DataFrame({'Feature': X_train_cb.columns, 'Importance': importances})
feat_df = feat_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 7))
top15 = feat_df.head(15).sort_values(by='Importance', ascending=True)
plt.barh(top15['Feature'], top15['Importance'], color='#3498db', edgecolor='black')
plt.title('Top 15 Predictive Features (CatBoost Importance)', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance Gain')
plt.tight_layout()
plt.show()

feat_df.head(10)

## 13. Final Model Retraining on Full Dataset
CatBoost achieved the highest ROC-AUC score (**0.7227**). We retrain the model on 100% of the training data (`data/loan_train.csv`).

In [ ]:
full_clean = clean_dataset(df_train, is_train=True)
X_full = full_clean.drop(columns=['loan_status'])
y_full = full_clean['loan_status'].astype(int)

full_preprocessor = LoanPreprocessor().fit(X_full)
X_full_cb = full_preprocessor.transform_catboost(X_full)
full_pool = Pool(X_full_cb, y_full, cat_features=cat_indices)

best_iter = cb_model.get_best_iteration()
final_iterations = (best_iter + 50) if best_iter and best_iter > 0 else 300

final_catboost = CatBoostClassifier(
    iterations=final_iterations,
    depth=7,
    learning_rate=0.05,
    eval_metric='AUC',
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=100
)
final_catboost.fit(full_pool)
print('Final CatBoost model retrained successfully!')

## 14 & 15. Test Set Prediction & Submission Generation
We transform `loan_test.csv` through the exact same pipeline and generate `outputs/loan_submission.csv`.

In [ ]:
test_clean = clean_dataset(df_test, is_train=False)
test_transformed = full_preprocessor.transform_catboost(test_clean)

test_probs = final_catboost.predict_proba(test_transformed)[:, 1]
test_preds = (test_probs >= 0.5).astype(int)

submission_df = pd.DataFrame({
    'id': df_test['id'],
    'loan_status': test_preds
})

sub_path = '../outputs/loan_submission.csv' if os.path.exists('../outputs') else 'outputs/loan_submission.csv'
os.makedirs(os.path.dirname(sub_path), exist_ok=True)
submission_df.to_csv(sub_path, index=False)

print(f'Submission shape: {submission_df.shape}')
print('Submission value counts:')
print(submission_df['loan_status'].value_counts())
submission_df.head(10)